# SentinelPay: Data Preprocessing & Leakage Prevention
## Notebook 02 — Train/Test Split, Scaling, and SMOTE on Training Split Only

### 1. Preventing Data Leakage
A common flaw in fraud detection models is applying oversampling (e.g. SMOTE) or feature scaling across the entire dataset before splitting. This causes synthetic patterns or test distributions to leak into the training fold.

**Hygiene Rules:**
1. Perform Stratified Train/Test split first (80/20).
2. Fit Scaler (RobustScaler) on Train split only.
3. Apply SMOTE to Train split only; leave Test split pristine.

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/raw/sentinelpay_benchmark_transactions.csv')
features = ['amount', 'distance', 'time_delta', 'merchant_risk', 'device_trust', 'velocity_1h', 'velocity_24h', 'hour_of_day', 'is_weekend']

X = df[features]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

# Fit scaler on Train ONLY
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE on Train ONLY
smote = SMOTE(sampling_strategy=0.25, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f"Before SMOTE Train Fraud: {y_train.sum()} | After SMOTE: {y_train_res.sum()}")